In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
# Repository Clone
%cd /kaggle/working
!git clone https://github.com/LakshayBaijal/Computer-Vision_Assignments_Lakshay.git
%cd /kaggle/working/Computer-Vision_Assignments_Lakshay/Q1

/kaggle/working
Cloning into 'Computer-Vision_Assignments_Lakshay'...
remote: Enumerating objects: 30, done.
remote: Counting objects: 100% (30/30), done.
remote: Compressing objects: 100% (26/26), done.
remote: Total 30 (delta 1), reused 27 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (30/30), 344.64 KiB | 3.45 MiB/s, done.
Resolving deltas: 100% (1/1), done.
/kaggle/working/Computer-Vision_Assignments_Lakshay/Q1


In [12]:
from pathlib import Path

# Find Q1 folder
q1_candidates = list(Path("/kaggle/working").rglob("Q1/train.py"))
assert q1_candidates, "Q1/train.py not found. Clone your repo first."
REPO = q1_candidates[0].parent

# Find dataset root (must contain img + annots)
data_root = None
for d in Path("/kaggle/input").rglob("*"):
    if d.is_dir() and (d / "img").exists() and (d / "annots").exists():
        data_root = d
        break
assert data_root is not None, "Dataset root with img/ and annots/ not found in /kaggle/input"

print("REPO:", REPO)
print("DATA:", data_root)

REPO: /kaggle/working/Computer-Vision_Assignments_Lakshay/Q1
DATA: /kaggle/input/datasets/lakshaybaijal/q1-dataset


In [18]:
%pip install -q einops matplotlib "numpy>=1.26,<2.2" opencv-python PyYAML tqdm shapely Pillow

Note: you may need to restart the kernel to use updated packages.


In [19]:
import yaml, time
from pathlib import Path

run_tag = time.strftime("%Y%m%d_%H%M%S")
cfg_in = REPO / "config" / "st.yaml"
cfg_out = Path(f"/kaggle/working/run3_obb_direct_{run_tag}.yaml")

with open(cfg_in, "r") as f:
    cfg = yaml.safe_load(f)

cfg["dataset_params"]["root_dir"] = str(data_root)

cfg["train_params"]["use_angle"] = True
cfg["train_params"]["num_epochs"] = 15
cfg["train_params"]["lr"] = 5e-4
cfg["train_params"]["lr_steps"] = [10, 13]

# Use relative checkpoint folder to avoid save-path issues
cfg["train_params"]["task_name"] = f"checkpoints_run3_obb_direct_{run_tag}"
cfg["train_params"]["ckpt_name"] = f"run3_obb_direct_{run_tag}.pth"

with open(cfg_out, "w") as f:
    yaml.safe_dump(cfg, f, sort_keys=False)

print("Config:", cfg_out)
print("Train params:", cfg["train_params"])

Config: /kaggle/working/run3_obb_direct_20260405_122745.yaml
Train params: {'task_name': 'checkpoints_run3_obb_direct_20260405_122745', 'seed': 1111, 'infer_seed': 1122, 'acc_steps': 1, 'num_epochs': 15, 'lr_steps': [10, 13], 'lr': 0.0005, 'ckpt_name': 'run3_obb_direct_20260405_122745.pth', 'use_angle': True}


In [20]:
import os, sys

os.chdir(str(REPO))
sys.path.insert(0, str(REPO))

from train import train

class Args:
    config_path = str(cfg_out)
    root_dir = str(data_root)

train(Args())
print("Training finished.")

{'dataset_params': {'root_dir': '/kaggle/input/datasets/lakshaybaijal/q1-dataset', 'num_classes': 2}, 'model_params': {'im_channels': 3, 'aspect_ratios': [0.5, 1, 2], 'scales': [128, 256, 512], 'min_im_size': 600, 'max_im_size': 1000, 'backbone_out_channels': 512, 'fc_inner_dim': 1024, 'rpn_bg_threshold': 0.3, 'rpn_fg_threshold': 0.7, 'rpn_nms_threshold': 0.7, 'rpn_train_prenms_topk': 12000, 'rpn_test_prenms_topk': 6000, 'rpn_train_topk': 2000, 'rpn_test_topk': 300, 'rpn_batch_size': 256, 'rpn_pos_fraction': 0.5, 'roi_iou_threshold': 0.5, 'roi_low_bg_iou': 0.0, 'roi_pool_size': 7, 'roi_nms_threshold': 0.3, 'roi_topk_detections': 100, 'roi_score_threshold': 0.05, 'roi_batch_size': 128, 'roi_pos_fraction': 0.25, 'bbox_reg_weights': [10.0, 10.0, 5.0, 5.0, 1.0]}, 'train_params': {'task_name': 'checkpoints_run3_obb_direct_20260405_122745', 'seed': 1111, 'infer_seed': 1122, 'acc_steps': 1, 'num_epochs': 15, 'lr_steps': [10, 13], 'lr': 0.0005, 'ckpt_name': 'run3_obb_direct_20260405_122745.pth

100%|██████████| 181/181 [02:36<00:00,  1.16it/s]


Finished epoch 0
Checkpoint saved: checkpoints_run3_obb_direct_20260405_122745/tv_frcnn_r50fpn_run3_obb_direct_20260405_122745.pth
RPN Classification Loss : 0.1675 | RPN Localization Loss : 0.1079 | FRCNN Classification Loss : 0.9533 | FRCNN Localization Loss : 0.4096


100%|██████████| 181/181 [02:37<00:00,  1.15it/s]


Finished epoch 1
Checkpoint saved: checkpoints_run3_obb_direct_20260405_122745/tv_frcnn_r50fpn_run3_obb_direct_20260405_122745.pth
RPN Classification Loss : 0.0657 | RPN Localization Loss : 0.0898 | FRCNN Classification Loss : 0.8022 | FRCNN Localization Loss : 0.4103


...

Finished epoch 12
Checkpoint saved: checkpoints_run3_obb_direct_20260405_122745/tv_frcnn_r50fpn_run3_obb_direct_20260405_122745.pth
RPN Classification Loss : 0.0156 | RPN Localization Loss : 0.0461 | FRCNN Classification Loss : 0.4920 | FRCNN Localization Loss : 0.2945


100%|██████████| 181/181 [02:38<00:00,  1.15it/s]


Finished epoch 13
Checkpoint saved: checkpoints_run3_obb_direct_20260405_122745/tv_frcnn_r50fpn_run3_obb_direct_20260405_122745.pth
RPN Classification Loss : 0.0144 | RPN Localization Loss : 0.0443 | FRCNN Classification Loss : 0.4740 | FRCNN Localization Loss : 0.2870


100%|██████████| 181/181 [02:37<00:00,  1.15it/s]


Finished epoch 14
Checkpoint saved: checkpoints_run3_obb_direct_20260405_122745/tv_frcnn_r50fpn_run3_obb_direct_20260405_122745.pth
RPN Classification Loss : 0.0130 | RPN Localization Loss : 0.0423 | FRCNN Classification Loss : 0.4585 | FRCNN Localization Loss : 0.2792
Done Training...
Training finished.


In [21]:
import glob, os
from pathlib import Path

# Expected directory (relative to REPO)
expected_dir = REPO / cfg["train_params"]["task_name"]
cand = sorted(glob.glob(str(expected_dir / "*.pth")))

# Fallback search
if not cand:
    cand = sorted(glob.glob("/kaggle/working/**/*.pth", recursive=True), key=os.path.getmtime)

assert cand, "No .pth checkpoint found after training."
print("Checkpoint count:", len(cand))
print("Latest:", cand[-1])

Checkpoint count: 1
Latest: /kaggle/working/Computer-Vision_Assignments_Lakshay/Q1/checkpoints_run3_obb_direct_20260405_122745/tv_frcnn_r50fpn_run3_obb_direct_20260405_122745.pth


In [11]:
import os, json, shutil, time
from pathlib import Path

latest = cand[-1]
export_tag = time.strftime("%Y%m%d_%H%M%S")
export_dir = Path(f"/kaggle/working/EXPORT_Q1_RUN3_OBB_DIRECT_{export_tag}")
export_dir.mkdir(parents=True, exist_ok=True)

final_pth = export_dir / f"Q1_RUN3_OBB_DIRECT_final_{export_tag}.pth"
shutil.copy2(latest, final_pth)

manifest = {
    "run": "Q1_RUN3_OBB_DIRECT",
    "latest_checkpoint": latest,
    "config_used": str(cfg_out),
    "data_root": str(data_root),
}
with open(export_dir / f"Q1_RUN3_OBB_DIRECT_manifest_{export_tag}.json", "w") as f:
    json.dump(manifest, f, indent=2)

zip_path = shutil.make_archive(f"/kaggle/working/Q1_RUN3_OBB_DIRECT_EXPORT_{export_tag}", "zip", str(export_dir))
print("Export zip:", zip_path)

Found .pth files: 2
 - /kaggle/working/EXPORT_Q1_RUN2_AABB_HP2/Q1_RUN2_AABB_HP2_final.pth
 - /kaggle/working/Q1_run2_aabb_hp2_ckpts/tv_frcnn_r50fpn_aabb_hp2.pth

Using checkpoint: /kaggle/working/Q1_run2_aabb_hp2_ckpts/tv_frcnn_r50fpn_aabb_hp2.pth

Created zip: /kaggle/working/Q1_RUN2_AABB_HP2_EXPORT_20260405_121149.zip
Size (MB): 146.8490390777588


/kaggle/working/Q1_RUN2_AABB_HP2_EXPORT_20260405_121149.zip

If browser blocks popup, click the link above manually.


In [ ]:
from IPython.display import display, FileLink, HTML
import os

zip_name = os.path.basename(zip_path)
display(FileLink(zip_path))
display(HTML(f'<a href="/files/{zip_name}" target="_blank">Download {zip_name}</a>'))
display(HTML(f'<script>window.open("/files/{zip_name}", "_blank");</script>'))
print("If popup blocked, click the link above manually.")